<a href="https://colab.research.google.com/github/jabirjabzz/AI-Agents/blob/main/fooocus_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q diffusers transformers accelerate controlnet_aux

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.1/509.1 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 114.7 MB/s eta 0:00:00


In [4]:
from diffusers import StableDiffusionControlNetImg2ImgPipeline, ControlNetModel, UniPCMultistepScheduler
import torch

# Load the ControlNet model for canny edges
controlnet = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)

# Initialize the pipeline
pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

# Speed up with UniPC scheduler
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe.to(device)

print(f"Pipeline loaded on {device}")

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_

config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pipeline loaded on cpu


In [8]:
import torch
import cv2
import numpy as np
import os
from PIL import Image

# Try to import torch_xla for TPU support
tpu_available = False
try:
    import torch_xla.core.xla_model as xm
    import torch_xla.device as xd
    tpu_available = True
except ImportError:
    pass

# --- SET YOUR REFERENCE IMAGE HERE ---
REF_IMAGE_PATH = "/content/sample_data/Hero_image.png" # Path to your realistic box image

# Configuration dictionary with focus on realism and tape
PROMPTS = {
    "D1": {
        "prompt": "extremely realistic product photo of a sealed cardboard shipping box, high-quality kraft paper texture, brown industrial packing tape sealed perfectly across the top, soft studio lighting, pure white background, sharp focus, 8k uhd",
        "negative": "bad anatomy, text, logo, blurry, low quality, distorted, cartoon, 3d render, illustration, messy tape",
        "strength": 0.60,
        "control": 0.95
    },
    "D2": {
        "prompt": "realistic kraft cardboard box, professional lighting, brown tape on top, pure white background, high resolution, product photography",
        "negative": "text, watermark, logo, blurry, drawing, fake texture",
        "strength": 0.70,
        "control": 0.85
    },
    "D3": {
        "prompt": "realistic open cardboard box, kraft paper interior and exterior, flaps open, studio lighting, white background, 8k",
        "negative": "closed box, messy, text, logo, cartoon",
        "strength": 0.65,
        "control": 0.90
    }
}

def get_canny(image_pil, low=100, high=200):
    img = np.array(image_pil.convert("RGB"))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, low, high)
    edges = np.stack([edges] * 3, axis=-1)
    return Image.fromarray(edges)

INPUT_DIR = "/content/sample_data/test_folder"
OUTPUT_DIR = "/content/sample_data/output_folder"
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Device detection
if tpu_available:
    device = xd.xr.device()
    print("Using TPU device")
elif torch.cuda.is_available():
    device = "cuda"
    print("Using CUDA device")
else:
    device = "cpu"
    print("Using CPU device")

# Load Reference if exists
ref_img = None
if os.path.exists(REF_IMAGE_PATH):
    ref_img = Image.open(REF_IMAGE_PATH).convert("RGB")
    print("Reference image loaded successfully.")

generator = torch.Generator(device='cpu').manual_seed(42)

for fname in sorted(os.listdir(INPUT_DIR)):
    if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue

    img_type = "D1"
    if "_D2" in fname.upper(): img_type = "D2"
    elif "_D3" in fname.upper(): img_type = "D3"

    cfg = PROMPTS[img_type]
    init_img = Image.open(os.path.join(INPUT_DIR, fname)).convert("RGB")

    # Preserve shape/size using Canny from the source image
    canny_control = get_canny(init_img)

    w, h = init_img.size
    scale = 768 / max(w, h)
    new_w, new_h = (int(w * scale) // 64 * 64, int(h * scale) // 64 * 64)

    init_img = init_img.resize((new_w, new_h), Image.LANCZOS)
    canny_control = canny_control.resize((new_w, new_h), Image.NEAREST)

    print(f"Processing [{img_type}] {fname} on {device}...")

    # Inference using the pipeline (assuming 'pipe' is initialized as ControlNetImg2Img)
    result = pipe(
        prompt=cfg["prompt"],
        negative_prompt=cfg["negative"],
        image=init_img,
        control_image=canny_control,
        strength=cfg["strength"],
        guidance_scale=8.0,
        num_inference_steps=30,
        controlnet_conditioning_scale=cfg["control"],
        generator=generator,
    ).images[0]

    result = result.resize((w, h), Image.LANCZOS)
    result.save(os.path.join(OUTPUT_DIR, f"enhanced_{fname}"))
    print(f"  Saved to: {OUTPUT_DIR}/enhanced_{fname}")

print("\n--- Processing Complete ---")

Using CPU device
Reference image loaded successfully.
Processing [D1] JGEC001_D1.png on cpu...


  0%|          | 0/18 [00:00<?, ?it/s]

  Saved to: /content/sample_data/output_folder/enhanced_JGEC001_D1.png
Processing [D2] JGEC001_D2.png on cpu...


  0%|          | 0/21 [00:00<?, ?it/s]

  Saved to: /content/sample_data/output_folder/enhanced_JGEC001_D2.png
Processing [D3] JGEC001_D3.png on cpu...


  0%|          | 0/19 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [5]:
PROMPTS = {
    "D1": {
        "prompt": (
            "professional product photography of a sealed cardboard shipping box, "
            "kraft paper texture, brown packing tape sealed across the top, "
            "soft studio lighting, pure white background, subtle drop shadow, "
            "photorealistic, 8k, highly detailed"
        ),
        "negative": (
            "mockup, 3d render, cartoon, illustration, drawing, artificial, "
            "text, watermark, logo, dimensions, markings, labels, people, hands, "
            "gray background, gradient background"
        ),
        "strength": 0.65,
        "control": 0.92
    },
    "D2": {
        "prompt": (
            "professional product photography of a sealed cardboard shipping box, "
            "kraft paper texture, packing tape on top, soft studio lighting, "
            "pure white background, clean surface with no text, photorealistic, 8k"
        ),
        "negative": (
            "mockup, 3d render, cartoon, illustration, text, watermark, logo, "
            "dimensions, markings, dimension lines, callouts, labels, arrows, "
            "measurements, people, hands, gray background"
        ),
        "strength": 0.75,
        "control": 0.85
    },
    "D3": {
        "prompt": (
            "professional product photography of an open cardboard shipping box "
            "with top flaps folded open, kraft paper texture, soft studio lighting, "
            "pure white background, subtle drop shadow, photorealistic, 8k"
        ),
        "negative": (
            "mockup, 3d render, cartoon, illustration, text, watermark, logo, "
            "dimensions, markings, labels, people, hands, gray background, "
            "closed lid, sealed top"
        ),
        "strength": 0.68,
        "control": 0.88
    }
}

def get_canny(image_pil, low=100, high=200):
    img = np.array(image_pil.convert("RGB"))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, low, high)
    edges = np.stack([edges] * 3, axis=-1)
    return Image.fromarray(edges)

In [7]:
# Check for CUDA availability automatically
device = "cuda" if torch.cuda.is_available() else "cpu"
generator = torch.Generator(device=device).manual_seed(42)

for fname in sorted(os.listdir(INPUT_DIR)):
    if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue

    img_type = "D1"
    if "_D2" in fname.upper():
        img_type = "D2"
    elif "_D3" in fname.upper():
        img_type = "D3"

    cfg = PROMPTS[img_type]
    init_img = Image.open(os.path.join(INPUT_DIR, fname)).convert("RGB")
    canny_img = get_canny(init_img)

    w, h = init_img.size
    scale = 768 / max(w, h)
    new_w = int(w * scale) // 64 * 64
    new_h = int(h * scale) // 64 * 64
    init_img = init_img.resize((new_w, new_h), Image.LANCZOS)
    canny_img = canny_img.resize((new_w, new_h), Image.NEAREST)

    print(f"[{img_type}] {fname} -> {new_w}x{new_h} on {device}")

    result = pipe(
        prompt=cfg["prompt"],
        negative_prompt=cfg["negative"],
        image=init_img,
        control_image=canny_img,
        strength=cfg["strength"],
        guidance_scale=7.5,
        num_inference_steps=25,
        controlnet_conditioning_scale=cfg["control"],
        generator=generator,
    ).images[0]

    result = result.resize((w, h), Image.LANCZOS)
    result.save(os.path.join(OUTPUT_DIR, f"enhanced_{fname}"))
    print(f"  Saved: enhanced_{fname}")

print(f"\n--- Done. Download from {OUTPUT_DIR} ---")


--- Done. Download from /content/sample_data/output_folder ---


# New Section

In [1]:
!pip install -q diffusers transformers accelerate controlnet_aux opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 7.3 MB/s eta 0:00:00


In [2]:
import torch
import cv2
import numpy as np
import os
from PIL import Image
from diffusers import (
    StableDiffusionControlNetImg2ImgPipeline,
    ControlNetModel,
    UniPCMultistepScheduler
)

# ✅ Recommended: GPU > CPU > TPU (TPU not supported by diffusers)
if torch.cuda.is_available():
    device = "cuda"
    dtype  = torch.float16
    print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    dtype  = torch.float32
    print("⚠️ No GPU found — running on CPU (will be slow)")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


✅ Using GPU: Tesla T4


In [11]:
from transformers import CLIPVisionModelWithProjection, CLIPImageProcessor
import torch.nn.functional as F

# Load CLIP vision encoder (same one IP-Adapter uses internally)
clip_encoder = CLIPVisionModelWithProjection.from_pretrained(
    "openai/clip-vit-large-patch14",
    torch_dtype=dtype
).to(device)

clip_processor = CLIPImageProcessor.from_pretrained("openai/clip-vit-large-patch14")

def get_ref_blend(init_pil, ref_pil, alpha=0.25):
    """
    Blend reference image texture/color into the init image.
    alpha=0 means no reference, alpha=1 means full reference.
    Keep alpha low (0.2–0.35) to preserve box structure.
    """
    ref_resized = ref_pil.resize(init_pil.size, Image.LANCZOS)
    return Image.blend(init_pil, ref_resized, alpha=alpha)

def get_clip_prompt_embeds(ref_pil, text_prompt, pipe, device, dtype):
    """
    Encode reference image with CLIP and concatenate with text embeddings.
    This gives the pipeline both text + visual reference guidance.
    """
    # Encode text
    text_inputs = pipe.tokenizer(
        text_prompt,
        padding="max_length",
        max_length=pipe.tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt"
    )
    with torch.no_grad():
        text_embeds = pipe.text_encoder(
            text_inputs.input_ids.to(device)
        )[0]  # shape: [1, 77, 768]

    # Encode reference image with CLIP
    clip_inputs = clip_processor(images=ref_pil, return_tensors="pt").to(device)
    with torch.no_grad():
        image_features = clip_encoder(**clip_inputs).image_embeds  # [1, 768]

    # Project image features to text embedding space and append as extra tokens
    image_tokens = image_features.unsqueeze(1).expand(-1, 4, -1)   # [1, 4, 768]
    image_tokens = F.pad(image_tokens, (0, 0, 0, 0))               # keep shape

    # Concatenate: text gets visual context appended
    combined = torch.cat([text_embeds, image_tokens], dim=1)        # [1, 81, 768]

    # Truncate back to max length (77 tokens)
    combined = combined[:, :pipe.tokenizer.model_max_length, :]     # [1, 77, 768]

    return combined

print("✅ CLIP reference encoder ready")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

✅ CLIP reference encoder ready


In [13]:
REF_IMAGE_PATH = "/content/sample_data/Hero_image.png"

ref_img = None
if os.path.exists(REF_IMAGE_PATH):
    ref_img = Image.open(REF_IMAGE_PATH).convert("RGB")
    print(f"✅ Reference image loaded: {ref_img.size}")
else:
    print("⚠️ Reference image NOT found — IP-Adapter will be disabled for this run")
    print(f"   Expected path: {REF_IMAGE_PATH}")

✅ Reference image loaded: (1448, 1086)


In [14]:
PROMPTS = {
    # D1 — Sealed box, tape on top
    "D1": {
        "prompt": (
            "professional product photography of a sealed cardboard shipping box, "
            "kraft paper texture, brown packing tape sealed across the top, "
            "soft studio lighting, pure white background, subtle drop shadow, "
            "photorealistic, 8k, highly detailed"
        ),
        "negative": (
            "mockup, 3d render, cartoon, illustration, drawing, text, watermark, "
            "logo, dimensions, labels, people, hands, gray background, gradient"
        ),
        "strength": 0.65,   # how much to change the input image (0=none, 1=ignore input)
        "control": 0.92     # how much to follow the canny edges
    },
    # D2 — Sealed box, minimal markings
    "D2": {
        "prompt": (
            "professional product photography of a sealed cardboard shipping box, "
            "kraft paper texture, packing tape on top, soft studio lighting, "
            "pure white background, clean surface with no text, photorealistic, 8k"
        ),
        "negative": (
            "mockup, 3d render, cartoon, illustration, text, watermark, logo, "
            "dimension lines, callouts, arrows, measurements, people, hands, "
            "gray background"
        ),
        "strength": 0.75,
        "control": 0.85
    },
    # D3 — Open box, flaps up
    "D3": {
        "prompt": (
            "professional product photography of an open cardboard shipping box "
            "with top flaps folded open, kraft paper texture, soft studio lighting, "
            "pure white background, subtle drop shadow, photorealistic, 8k"
        ),
        "negative": (
            "mockup, 3d render, cartoon, illustration, text, watermark, logo, "
            "dimensions, labels, people, hands, gray background, closed lid, sealed top"
        ),
        "strength": 0.68,
        "control": 0.88
    }
}

In [15]:
def get_canny(image_pil, low=100, high=200):
    """Extract Canny edges from a PIL image, returned as a 3-channel PIL image."""
    img   = np.array(image_pil.convert("RGB"))
    gray  = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, low, high)
    edges = np.stack([edges] * 3, axis=-1)   # make 3-channel for ControlNet
    return Image.fromarray(edges)


def resize_to_64(image_pil, max_size=768):
    """Resize image so longest side = max_size, both dims divisible by 64."""
    w, h   = image_pil.size
    scale  = max_size / max(w, h)
    new_w  = int(w * scale) // 64 * 64
    new_h  = int(h * scale) // 64 * 64
    return image_pil.resize((new_w, new_h), Image.LANCZOS), (w, h)

In [25]:
INPUT_DIR  = "/content/sample_data/test_folder"
OUTPUT_DIR = "/content/sample_data/output_folder"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Ensure the generator is on the correct device
generator = torch.Generator(device=device).manual_seed(42)

image_files = [
    f for f in sorted(os.listdir(INPUT_DIR))
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
]
print(f"Found {len(image_files)} image(s)\n")

for fname in image_files:
    fname_upper = fname.upper()
    if   "_D3" in fname_upper: img_type = "D3"
    elif "_D2" in fname_upper: img_type = "D2"
    else:                       img_type = "D1"

    cfg      = PROMPTS[img_type]
    init_img = Image.open(os.path.join(INPUT_DIR, fname)).convert("RGB")
    init_resized, original_size = resize_to_64(init_img)

    # Apply reference image blending for texture
    if ref_img is not None:
        init_blended = get_ref_blend(init_resized, ref_img, alpha=0.3)
    else:
        init_blended = init_resized

    canny_img = get_canny(init_resized)

    print(f"Processing [{img_type}] {fname}...")

    # Use a simple dummy embedding to bypass the IP-Adapter check without unloading
    # SD 1.5 expects 768 hidden dims. Batch size 2 for CFG (pos/neg).
    dummy_embeds = [torch.zeros((2, 1, 768), device=device, dtype=dtype)]

    result = pipe(
        prompt=cfg["prompt"],
        negative_prompt=cfg["negative"],
        image=init_blended,
        control_image=canny_img,
        strength=cfg["strength"],
        guidance_scale=8.0,
        num_inference_steps=30,
        controlnet_conditioning_scale=cfg["control"],
        generator=generator,
        ip_adapter_image_embeds=dummy_embeds
    ).images[0]

    result = result.resize(original_size, Image.LANCZOS)
    out_path = os.path.join(OUTPUT_DIR, f"enhanced_{fname}")
    result.save(out_path)
    print(f"  ✅ Saved: {out_path}\n")

print("─── All done ───")

Found 12 image(s)

Processing [D1] JGEC001_D1.png...


  0%|          | 0/19 [00:00<?, ?it/s]

  ✅ Saved: /content/sample_data/output_folder/enhanced_JGEC001_D1.png

Processing [D2] JGEC001_D2.png...


  0%|          | 0/22 [00:00<?, ?it/s]

  ✅ Saved: /content/sample_data/output_folder/enhanced_JGEC001_D2.png

Processing [D3] JGEC001_D3.png...


...

  ✅ Saved: /content/sample_data/output_folder/enhanced_JGEC004_D1.png

Processing [D2] JGEC004_D2.png...


  0%|          | 0/22 [00:00<?, ?it/s]

  ✅ Saved: /content/sample_data/output_folder/enhanced_JGEC004_D2.png

Processing [D3] JGEC004_D3.png...


  0%|          | 0/20 [00:00<?, ?it/s]

  ✅ Saved: /content/sample_data/output_folder/enhanced_JGEC004_D3.png

─── All done ───


# New Section